
# Sheared Lennard–Jones (Lees–Edwards) with JAX MD

Simulate a small 3D Lennard–Jones (LJ) fluid under a constant shear rate using the `space.shearing` (Lees–Edwards / tilted box) transformation and **fractional coordinates**.

What this notebook demonstrates:

1. Construction of a time–dependent sheared simulation cell with optional shear remapping (keeping |gamma| < 0.5).
2. Use of fractional coordinates so particle positions remain in [0,1)^3 while the real-space cell tilts in time.
3. On-the-fly neighbor list updates as the box deforms.
4. Brownian dynamics (overdamped Langevin) at target temperature kT.


In [ ]:

import pathlib
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from jax_md import space, partition, simulate

jax.config.update("jax_enable_x64", False)

# Output directory for any saved artifacts (images, data, etc.)
outdir = pathlib.Path("_artifacts")
outdir.mkdir(parents=True, exist_ok=True)

print("JAX devices:", jax.devices())


In [ ]:
# Utility helpers -----------------------------------------------------------

def check_overlaps(R, sigma, box_of, t=0.0):
    """Return True if no pair of particles is closer than r_vdw in real space.

    r_vdw = 2**(1/6) * sigma (LJ minimum)
    Parameters
    ----------
    R : (N, 3) fractional coordinates in [0,1)^3
    sigma : LJ length scale
    box_of : callable mapping time -> box matrix H(t)
    t : float, time used to obtain the instantaneous (possibly sheared) box
    """
    H = box_of(t=t)                      # 3x3 deformation (tilted) matrix
    R_real = space.transform(H, R)       # fractional -> real coordinates
    d = jnp.linalg.norm(R_real[:, None] - R_real[None, :], axis=-1)
    d = d + jnp.eye(R.shape[0]) * 1e10   # mask self distances
    rvdw = 2**(1/6) * sigma        # van der Waals distance
    return jnp.all(d >= rvdw)


def create_initial_positions(key, N, sigma, box_of):
    """Generate random fractional positions with a crude overlap rejection.

    Accept configurations only if all pair distances exceed r_vdw
    (LJ minimum) in real space at the current box.
    """
    tries = 0
    while True:
        tries += 1
        R = jax.random.uniform(key, (N, 3), minval=0.0, maxval=1.0)
        if bool(check_overlaps(R, sigma, box_of)):
            print(f"Generated positions with no overlaps in {tries} attempt(s).")
            return R
        key, _ = jax.random.split(key)



### 1. Parameters
We first define simulation, thermodynamic, and interaction parameters. The shear deformation is specified by a constant strain rate `shear_rate`; if `remap=True`, the instantaneous shear strain `gamma` is periodically wrapped into a principal window (similar to Lees–Edwards sliding bricks) so that the cell does not become arbitrarily tilted.


In [ ]:

# --- Simulation Parameters -------------------------------------------------
N = 250                 # number of particles
steps = 1000             # total integration steps
dt = 1e-3               # time step

# Box dimensions (orthogonal reference box before shear tilt is applied)
Lx, Ly, Lz = 10.0, 10.0, 10.0

# Constant shear: gamma(t) = shear_rate * t. remap keeps gamma in [-0.5, 0.5)
shear_rate = 100        # shear strain rate (1 / time units)
remap = True            # periodically wrap accumulated shear

# Thermostat / LJ parameters
kT = 1.0                # target temperature for Brownian dynamics noise
sigma = .5             # LJ sigma
epsilon = .5           # LJ epsilon
rc = 2.5 * sigma      # LJ cutoff radius
skin = 0.4 * rc         # neighbor list skin (buffer) for rebuilds

plot_every = 50         # interval (steps) for snapshot collection (0 => disable)



### 2. Shearing space construction
We build a sheared (tilted) periodic space using `space.shearing`. Because we keep positions in fractional coordinates, the integrator operates in a static unit cube while the real-space cell matrix `H(t)` evolves with time. Distances and shifts are always computed wrt the instantaneous tilted box when we pass `t` forward.


In [ ]:

# --- Shearing space (fractional coords) ------------------------------------
# B is the *reference* (unsheared) box matrix; shear modifies off-diagonal terms.
B = jnp.diag(jnp.array([Lx, Ly, Lz], dtype=jnp.float32))

# space.shearing returns displacement / shift functions that are aware of the
# time-dependent tilted box implementing Lees–Edwards like boundary conditions.
# Using fractional_coordinates=True means state.position always lives in [0,1)^3
# and we convert to real space only for analysis / visualization.
disp, shift, box_of = space.shearing(
    B, shear_rate=shear_rate, fractional_coordinates=True, remap=remap
)
metric = space.metric(disp)  # distance function consistent with sheared geometry

print(f"Using shearing space with shear_rate={shear_rate}, remap={remap}")



### 3. Initialization
We sample random fractional positions with a naive overlap rejection (no pair closer than r_vdw = 2**(1/6)*sigma). This keeps the initial energy reasonable without needing a minimization step. Real-space coordinates are obtained via `space.transform` using the current box matrix `H(t)`.


In [ ]:

key = jax.random.PRNGKey(0)
R0 = create_initial_positions(key, N, sigma, box_of)
H0 = box_of(t=0.0)
R0_real = space.transform(H0, R0)

print("R0 (fractional) shape:", R0.shape)
print("R0 (real) range x:", float(jnp.min(R0_real[:,0])), "->", float(jnp.max(R0_real[:,0])))



### 4. Neighbor list
We construct an ordered-sparse neighbor list (i<j) in fractional coordinates. The sheared displacement function is time-dependent, so during the dynamics loop we must call `neighbor_fn.update(..., box=H(t))` every step (or when needed). Safety flags trigger reallocation.


In [ ]:

neighbor_fn = partition.neighbor_list(
    disp,
    box=box_of(t=0.0),                # physical box at build time
    r_cutoff=rc,
    dr_threshold=skin/2.0,
    fractional_coordinates=True,
    format=partition.OrderedSparse,   # (i<j) pairs
)

nbrs = neighbor_fn.allocate(R0, box=box_of(t=0.0))
print("Neighbor list allocated. Format=OrderedSparse, pairs:", int(nbrs.idx.shape[1]))



### 5. Potential energy function
We define a shifted Lennard–Jones potential (energy shifted so that V(rc)=0). Forces are not smoothed at the cutoff; for smoother dynamics you could switch to a force-shifted or tapered potential. The metric uses the instantaneous shear at time `t`.


In [ ]:

@jax.jit
def lj_shifted(dr):
    """Standard 12-6 LJ with energy shifted to zero at rc (no force smoothing)."""
    inv = sigma / dr
    inv6 = inv**6
    inv12 = inv6**2
    v = 4.0 * epsilon * (inv12 - inv6)
    invc = sigma / rc
    v_shift = 4.0 * epsilon * (invc**12 - invc**6)
    return jnp.where(dr < rc, v - v_shift, 0.0)

@jax.jit
def energy_fn(R, *, neighbor_idx, t=0.0):
    """Total shifted LJ energy using current sheared metric.

    Padded (sentinel) pairs are assigned a large dummy displacement BEFORE
    taking norms so that autograd never sees zero vectors from duplicate
    (clamped) indices, avoiding NaNs in force initialization.
    """
    i, j = neighbor_idx  # (M,)
    N = R.shape[0]
    valid = (i < N) & (j < N)
    # Clamp for safe gather.
    safe_i = jnp.minimum(i, N - 1)
    safe_j = jnp.minimum(j, N - 1)
    # Displacements.
    dR = disp(R[safe_i], R[safe_j], t=t)  # (M,3)
    # Replace invalid displacements with large dummy vector so norm >> rc.
    dummy = jnp.array([rc * 10.0, 0.0, 0.0], dtype=dR.dtype)
    dR = jnp.where(valid[:, None], dR, dummy)
    dr = jnp.linalg.norm(dR, axis=-1)
    dr = jnp.clip(dr, 1e-6, None)
    e = lj_shifted(dr)
    return jnp.sum(e)



### 6. Thermostat and integrator
We build a Brownian dynamics (overdamped Langevin) integrator. Because positions are fractional, the integrator keeps them wrapped while the physical cell tilt is accounted for through the displacement/shift functions.


In [ ]:
# Build Brownian dynamics (overdamped Langevin) integrator; positions evolve
# under overdamped dynamics with stochastic noise (no explicit velocities).
# gamma is the friction coefficient.
gamma = 0.5
init_fn, apply_fn = simulate.brownian(energy_fn, shift, dt, kT, gamma=gamma)

state = init_fn(jax.random.PRNGKey(1), R0)
print("State fields:", state._fields if hasattr(state, "_fields") else type(state))


### 7. Time integration loop
Each step:
1. Advance the integrator with current shear time `t`.
2. Update (and possibly reallocate) the neighbor list using the instantaneous box `H(t)`.
3. Record potential energy and (optionally) a snapshot for visualization at intervals.


In [ ]:

import time

Es = []
traj_samples = []  # (t, real_positions, H) tuples for visualization

# Temperature trace (BD has no kinetic temperature; record target kT as reference)
Ts = []
ts = []

# Timing accumulators (total) and per-step lists
_timing_totals = {
    'integrate': 0.0,
    'neighbor_update': 0.0,
    'reallocate': 0.0,
    'energy': 0.0,
    'snapshot': 0.0,
}
_timing_steps = {k: [] for k in _timing_totals.keys()}

for step in range(steps):
    t = (step + 1) * dt

    # Integrator step -------------------------------------------------------
    t0 = time.perf_counter()
    state = apply_fn(state, neighbor_idx=nbrs.idx, t=t)
    jax.block_until_ready(state.position)
    dt_op = time.perf_counter() - t0
    _timing_totals['integrate'] += dt_op
    _timing_steps['integrate'].append(dt_op)

    # Neighbor list update --------------------------------------------------
    t0 = time.perf_counter()
    nbrs = neighbor_fn.update(state.position, nbrs, box=box_of(t=t))
    # jax.block_until_ready(nbrs.idx)
    dt_op = time.perf_counter() - t0
    _timing_totals['neighbor_update'] += dt_op
    _timing_steps['neighbor_update'].append(dt_op)

    # Possible reallocation -------------------------------------------------
    t0 = time.perf_counter()
    need_reallocate = any([
        bool(np.asarray(nbrs.did_buffer_overflow != 0).item()),
        bool(np.asarray(nbrs.cell_size_too_small != 0).item()),
        bool(np.asarray(nbrs.malformed_box != 0).item()),
    ])
    if need_reallocate:
        # Uncomment to see reallocation messages
        # print(f"Reallocating neighbor list at step {step} (t={t:.2f}) because of "
        #         f"buffer_overflow={nbrs.did_buffer_overflow}, "
        #         f"cell_size_too_small={nbrs.cell_size_too_small}, "
        #         f"malformed_box={nbrs.malformed_box}")
        nbrs = neighbor_fn.allocate(state.position, box=box_of(t=t))
        jax.block_until_ready(nbrs.idx)
    dt_op = time.perf_counter() - t0
    _timing_totals['reallocate'] += dt_op
    _timing_steps['reallocate'].append(dt_op)

    # Energy computation ----------------------------------------------------
    t0 = time.perf_counter()
    U = energy_fn(state.position, neighbor_idx=nbrs.idx, t=t)
    U = jax.block_until_ready(U)
    dt_op = time.perf_counter() - t0
    _timing_totals['energy'] += dt_op
    _timing_steps['energy'].append(dt_op)
    Es.append(float(U))

    # Record reference temperature -----------------------------------------
    Ts.append(float(kT))
    ts.append(float(t))

    # Check validity (positions or energy NaN) ------------------------------
    if jnp.any(jnp.isnan(state.position)) or jnp.isnan(U):
        raise ValueError("Simulation produced NaN positions or energy at step", step,
                         "decreasing dt or shear_rate may help.")

    # Snapshot (optional) ---------------------------------------------------
    took_snapshot = False
    t0 = time.perf_counter()
    if plot_every and (step % plot_every == 0 or step == steps - 1):
        H = box_of(t=t)
        R_real = np.array(space.transform(H, state.position % 1.0))
        traj_samples.append((t, R_real, np.array(H)))

        # BD does not track velocities; skip velocity profile accumulation.
        took_snapshot = True
        print(f"Step {step}  t = {t:.2f}  U = {U:.4f}", end="\t\t\r")
    dt_op = time.perf_counter() - t0
    if took_snapshot:
        _tming = _timing_totals  # keep alias to avoid accidental typos
        _timing_totals['snapshot'] += dt_op
        _timing_steps['snapshot'].append(dt_op)
    else:
        _timing_steps['snapshot'].append(0.0)

print("Done. Collected", len(traj_samples), "snapshots.")


### 8. Post-processing and visualization
This section summarizes common analyses and plots generated from the simulation:
- Snapshots (x–y projection) with a sheared box overlay.
- Shear field visualizations (box overlays, deformed grid).
- Potential energy trace.
- Temperature trace (reference kT recorded for BD).
- Timing summary for major operations.



#### 8.1 Snapshots (x–y projection)

This might take a few minutes to render depending on the number of particles and shear rate.


In [ ]:
# --- Visualization of snapshots with trajectories in 3 projections ----------

from matplotlib.patches import Circle
from matplotlib import animation
from IPython.display import Image, display

# van der Waals radius (minimum of LJ potential): r_vdw = 2**(1/6) * sigma
r_vdw = float(2**(1/6) * sigma)

def draw_cell_poly_axes(H, rows=(0, 1), cols=(0, 1)):
    """Return x,y for a 2D parallelogram spanned by columns cols projected on rows."""
    v1 = jnp.array([H[rows[0], cols[0]], H[rows[1], cols[0]]])
    v2 = jnp.array([H[rows[0], cols[1]], H[rows[1], cols[1]]])
    O = jnp.array([0.0, 0.0])
    P = jnp.stack([O, v1, v1 + v2, v2, O])
    return np.array(P[:, 0]), np.array(P[:, 1])

# Window wide enough for |gamma| <= 0.5 so tilted images remain visible (XY).
x_margin = 0.05 * max(Lx, Ly)
X_MIN = -0.5 * Ly - x_margin
X_MAX = Lx + 0.5 * Ly + x_margin
Y_MIN, Y_MAX = 0.0, Ly

# Conservative bounds for other projections.
x_margin_all = 0.05 * max(Lx, Ly, Lz)
XZ_X_MIN, XZ_X_MAX = 0.0 - x_margin_all, Lx + x_margin_all
Z_MIN, Z_MAX = 0.0, Lz
YZ_Y_MIN, YZ_Y_MAX = 0.0, Ly

if len(traj_samples) == 0:
    raise ValueError("No snapshots in traj_samples; set plot_every > 0 and rerun.")

# Prepare figure with three side-by-side projections: XY, XZ, YZ
fig, (ax_xy, ax_xz, ax_yz) = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for ax in (ax_xy, ax_xz, ax_yz):
    ax.set_aspect('equal', adjustable='box')

ax_xy.set_xlim(X_MIN - r_vdw, X_MAX + r_vdw)
ax_xy.set_ylim(Y_MIN - r_vdw, Y_MAX + r_vdw)
ax_xy.set_xlabel('x'); ax_xy.set_ylabel('y'); ax_xy.set_title('x–y')

ax_xz.set_xlim(XZ_X_MIN - r_vdw, XZ_X_MAX + r_vdw)
ax_xz.set_ylim(Z_MIN - r_vdw, Z_MAX + r_vdw)
ax_xz.set_xlabel('x'); ax_xz.set_ylabel('z'); ax_xz.set_title('x–z')

ax_yz.set_xlim(YZ_Y_MIN - r_vdw, YZ_Y_MAX + r_vdw)
ax_yz.set_ylim(Z_MIN - r_vdw, Z_MAX + r_vdw)
ax_yz.set_xlabel('y'); ax_yz.set_ylabel('z'); ax_yz.set_title('y–z')

# Initialize artists with the first frame
t0, R0_real, H0 = traj_samples[0]

# Pick highlighted particles deterministically and assign colors
highlight_idx = np.linspace(0, N - 1, min(10, N), dtype=int)
highlight_colors = {idx: plt.cm.tab10(i % 10) for i, idx in enumerate(highlight_idx.tolist())}

# Particle discs (all particles) in each projection
parts_xy, parts_xz, parts_yz = [], [], []
for i, pos in enumerate(R0_real):
    fc = highlight_colors.get(i, (0.4, 0.4, 0.4, 0.6))
    alpha = 0.9 if i in highlight_colors else 0.6
    pxy = Circle((float(pos[0]), float(pos[1])), r_vdw, facecolor=fc, edgecolor='none', alpha=alpha)
    pxz = Circle((float(pos[0]), float(pos[2])), r_vdw, facecolor=fc, edgecolor='none', alpha=alpha)
    pyz = Circle((float(pos[1]), float(pos[2])), r_vdw, facecolor=fc, edgecolor='none', alpha=alpha)
    ax_xy.add_patch(pxy); parts_xy.append(pxy)
    ax_xz.add_patch(pxz); parts_xz.append(pxz)
    ax_yz.add_patch(pyz); parts_yz.append(pyz)

# Box overlays (one per projection)
cell_xy, = ax_xy.plot(*draw_cell_poly_axes(H0, rows=(0, 1), cols=(0, 1)), 'k-', lw=1.0)
cell_xz, = ax_xz.plot(*draw_cell_poly_axes(H0, rows=(0, 2), cols=(0, 2)), 'k-', lw=1.0)
cell_yz, = ax_yz.plot(*draw_cell_poly_axes(H0, rows=(1, 2), cols=(1, 2)), 'k-', lw=1.0)

suptitle = fig.suptitle(f"t={t0:.2f}  gamma~{shear_rate * t0:.2f}")

# Precompute coordinates over time for trajectory lines
coords = np.array([R for (t, R, H) in traj_samples])  # (F, N, 3)
xs, ys, zs = coords[:, :, 0], coords[:, :, 1], coords[:, :, 2]

def init():
    return parts_xy + parts_xz + parts_yz + [cell_xy, cell_xz, cell_yz, suptitle]

def update(frame_idx):
    t, R_real, H = traj_samples[frame_idx]

    # Update particle positions
    for i in range(R_real.shape[0]):
        pos = R_real[i]
        parts_xy[i].center = (float(pos[0]), float(pos[1]))
        parts_xz[i].center = (float(pos[0]), float(pos[2]))
        parts_yz[i].center = (float(pos[1]), float(pos[2]))

    # Update box overlays
    cell_xy.set_data(*draw_cell_poly_axes(H, rows=(0, 1), cols=(0, 1)))
    cell_xz.set_data(*draw_cell_poly_axes(H, rows=(0, 2), cols=(0, 2)))
    cell_yz.set_data(*draw_cell_poly_axes(H, rows=(1, 2), cols=(1, 2)))

    suptitle.set_text(f"t={t:.2f}  gamma~{shear_rate * t:.2f}")
    return parts_xy + parts_xz + parts_yz + [cell_xy, cell_xz, cell_yz, suptitle]

anim = animation.FuncAnimation(
    fig,
    update,
    init_func=init,
    frames=len(traj_samples),
    interval=100,   # ms per frame
    blit=False,
)

# Save GIF (reuse existing code)
gif_path = outdir / "shear_xy.gif"
try:
    from matplotlib.animation import PillowWriter
    anim.save(gif_path.as_posix(), writer=PillowWriter(fps=10), dpi=150)
    print("Saved GIF to", gif_path)
except Exception as e:
    print("Failed to save with PillowWriter:", e)
    try:
        import imageio.v3 as iio
        # Fallback: render frames via the canvas and write with imageio
        frames = []
        for k in range(len(traj_samples)):
            update(k)
            fig.canvas.draw()
            frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
            frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
            frames.append(frame)
        iio.imwrite(gif_path.as_posix(), frames, duration=0.1, loop=0)
        print("Saved GIF via imageio to", gif_path)
    except Exception as e2:
        print("Failed to save GIF:", e2)

# Preview in notebook if available
plt.close(fig)
if gif_path.exists():
    display(Image(filename=str(gif_path)))

#### 8.2 Visualizing the shear field (more ways)
- Box outline overlays at multiple times to see tilt evolution.
- A deformed background grid to show the instantaneous shear mapping.
- Non-affine displacement quiver: deviation from purely affine shear relative to the initial configuration.


In [ ]:
# 8.2.a Box overlays at multiple times
sel = np.linspace(0, len(traj_samples)-1, 6, dtype=int) if len(traj_samples) > 0 else []
if len(sel) > 0:
    plt.figure(figsize=(5.5, 4))
    for idx in sel:
        t, R_real, H = traj_samples[idx]
        x, y = draw_cell_poly(H)
        plt.plot(x, y, '-', lw=1.5, label=f"t={t:.2f}")
    plt.gca().set_aspect('equal', adjustable='box')
    plt.xlim(X_MIN, X_MAX)
    plt.ylim(Y_MIN, Y_MAX)
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title('Sheared box overlays at selected times')
    plt.legend(ncol=2, fontsize=8)
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print("No snapshots to overlay; increase plot_every and rerun.")

In [ ]:
# 8.2.b Deformed background grid for a chosen time index
if len(traj_samples) > 0:
    idx = len(traj_samples) // 2
    t_mid, _, H_mid = traj_samples[idx]
    # Build a unit square grid in fractional coords and map with H_mid
    nlines = 12
    us = np.linspace(0, 1, nlines)
    plt.figure(figsize=(5.5, 4))
    # Vertical (u fixed, v varying)
    for u in us:
        Rf = np.stack([np.full_like(us, u), us, np.zeros_like(us)], axis=1)
        Rr = (np.array(H_mid) @ Rf.T).T
        plt.plot(Rr[:,0], Rr[:,1], color='0.75', lw=0.7)
    # Horizontal (v fixed, u varying)
    for v in us:
        Rf = np.stack([us, np.full_like(us, v), np.zeros_like(us)], axis=1)
        Rr = (np.array(H_mid) @ Rf.T).T
        plt.plot(Rr[:,0], Rr[:,1], color='0.75', lw=0.7)
    # Box outline
    x, y = draw_cell_poly(H_mid)
    plt.plot(x, y, 'k-', lw=1.2)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.xlim(X_MIN, X_MAX)
    plt.ylim(Y_MIN, Y_MAX)
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title(f'Deformed grid at t={t_mid:.2f}')
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print("No snapshots available; rerun with plot_every > 0.")

#### 8.4 Timing summary

In [ ]:

# --- Timing statistics -----------------------------------------------------
import pandas as pd

records = []
steps_executed = len(_timing_steps['integrate'])
for key in _timing_totals:
    total = _timing_totals[key]
    avg = total / steps_executed if steps_executed else 0.0
    mx = max(_timing_steps[key]) if _timing_steps[key] else 0.0
    records.append({'operation': key, 'total_s': total, 'avg_s': avg, 'max_s': mx})

df_timing = pd.DataFrame(records).sort_values('total_s', ascending=False)
display(df_timing)

slowest = df_timing.iloc[0]
print(f"Slowest operation: {slowest.operation} (total {slowest.total_s:.3f}s, avg {slowest.avg_s*1e3:.3f} ms/step)")
